# 05 — Gold Layer (with joins)

**Purpose:** business-ready aggregations built by joining clean trip data with TLC zone reference data.

**Inputs:**
- `silver_yellow_taxi` — fact table (~2.76M trips)
- `taxi_zone_lookup.csv` (raw) → ingested through Bronze + Silver inline → `silver_zone_lookup` (dim table, 265 rows)

**Outputs:**
- `gold_revenue_by_borough` — total revenue, trips, avg fare by pickup borough (single inner join)
- `gold_top_routes` — top 20 pickup→dropoff route pairs by trip count (self-join: dim joined twice with aliases)

**Join strategy:** the zone table is tiny (~265 rows), so we use Spark's `broadcast` hint — the entire dim table is shipped to every executor, eliminating shuffle. This is the right move whenever a dim table is small (~ <100 MB).

In [0]:
%run ./00_utils

Loaded helpers: LOG_TABLE, LOG_SCHEMA, log_pipeline_run, PIPELINE_NAME


In [0]:
import uuid
from pyspark.sql import functions as F
RUN_ID = str(uuid.uuid4())

In [0]:
ZONE_SOURCE        = "/Volumes/workspace/taxi/raw/taxi_zone_lookup.csv"
BRONZE_ZONES       = "workspace.taxi.bronze_zone_lookup"
SILVER_ZONES       = "workspace.taxi.silver_zone_lookup"
SILVER_TAXI        = "workspace.taxi.silver_yellow_taxi"
GOLD_BOROUGH       = "workspace.taxi.gold_revenue_by_borough"

try:
    # ---------- Bronze: ingest zone lookup CSV with lineage metadata ----------
    df_zones_raw = spark.read.csv(ZONE_SOURCE, header=True, inferSchema=True)
    df_zones_bronze = (df_zones_raw
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.lit(ZONE_SOURCE))
    )
    (df_zones_bronze.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(BRONZE_ZONES))
    log_pipeline_run("bronze_zones", df_zones_raw.count(), spark.table(BRONZE_ZONES).count(), "SUCCESS", RUN_ID)

    # ---------- Silver: snake_case rename ----------
    df_zones_silver = (spark.table(BRONZE_ZONES)
        .withColumnRenamed("LocationID", "location_id")
        .withColumnRenamed("Borough",    "borough")
        .withColumnRenamed("Zone",       "zone")
        # service_zone is already snake_case
    )
    (df_zones_silver.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(SILVER_ZONES))
    log_pipeline_run("silver_zones", df_zones_silver.count(), spark.table(SILVER_ZONES).count(), "SUCCESS", RUN_ID)

    # ---------- Gold #1: revenue by pickup borough (inner join with broadcast) ----------
    df_taxi  = spark.table(SILVER_TAXI)
    df_zones = spark.table(SILVER_ZONES)
    rows_in  = df_taxi.count()

    df_gold_borough = (df_taxi.alias("t")
        .join(
            F.broadcast(df_zones.alias("z")),    # broadcast = ship dim to every executor; no shuffle
            F.col("t.pickup_location_id") == F.col("z.location_id"),
            "inner"
        )
        .groupBy("z.borough")
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.sum("t.total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("t.fare_amount"),  2).alias("avg_fare"),
            F.round(F.avg("t.trip_distance"),2).alias("avg_trip_distance_miles")
        )
        .orderBy(F.desc("total_revenue"))
    )
    (df_gold_borough.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(GOLD_BOROUGH))

    log_pipeline_run("gold_revenue_by_borough", rows_in, spark.table(GOLD_BOROUGH).count(), "SUCCESS", RUN_ID)

except Exception as e:
    log_pipeline_run("gold_revenue_by_borough", -1, 0, "FAILED", RUN_ID, str(e))
    raise

/home/spark-8f6513b0-2114-4da1-9e9d-94/.ipykernel/2014/command-7532950260419597-1264827090:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(PIPELINE_NAME, run_id, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],


[bronze_zones] SUCCESS | rows_in=265 rows_out=265
[silver_zones] SUCCESS | rows_in=265 rows_out=265
[gold_revenue_by_borough] SUCCESS | rows_in=2,756,127 rows_out=8


In [0]:
GOLD_ROUTES = "workspace.taxi.gold_top_routes"

try:
    df_taxi  = spark.table(SILVER_TAXI)
    df_zones = spark.table(SILVER_ZONES)
    rows_in  = df_taxi.count()

    # Self-join trick: alias the same dim twice — once for pickup, once for dropoff.
    # This is the canonical pattern for "show me trips with NAMED start AND end locations".
    df_gold_routes = (df_taxi.alias("t")
        .join(F.broadcast(df_zones.alias("pu")),
              F.col("t.pickup_location_id")  == F.col("pu.location_id"),  "inner")
        .join(F.broadcast(df_zones.alias("do")),
              F.col("t.dropoff_location_id") == F.col("do.location_id"),  "inner")
        .groupBy(
            F.col("pu.borough").alias("pickup_borough"),
            F.col("pu.zone").alias("pickup_zone"),
            F.col("do.borough").alias("dropoff_borough"),
            F.col("do.zone").alias("dropoff_zone"),
        )
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.sum("t.total_amount"),         2).alias("total_revenue"),
            F.round(F.avg("t.trip_duration_minutes"),2).alias("avg_duration_min"),
            F.round(F.avg("t.trip_distance"),        2).alias("avg_distance_miles"),
        )
        .orderBy(F.desc("total_trips"))
        .limit(20)
    )
    (df_gold_routes.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(GOLD_ROUTES))

    log_pipeline_run("gold_top_routes", rows_in, spark.table(GOLD_ROUTES).count(), "SUCCESS", RUN_ID)

except Exception as e:
    log_pipeline_run("gold_top_routes", -1, 0, "FAILED", RUN_ID, str(e))
    raise

/home/spark-8f6513b0-2114-4da1-9e9d-94/.ipykernel/2014/command-7532950260419597-1264827090:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(PIPELINE_NAME, run_id, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],


[gold_top_routes] SUCCESS | rows_in=2,756,127 rows_out=20


In [0]:
print("=== Gold #1: Revenue by Pickup Borough ===")
spark.sql(f"SELECT * FROM {GOLD_BOROUGH}").show(truncate=False)

print("=== Gold #2: Top 20 Routes by Trip Count ===")
spark.sql(f"SELECT * FROM {GOLD_ROUTES}").show(truncate=False)

print("=== Pipeline log — full medallion trail ===")
spark.sql("""
    SELECT stage, rows_in, rows_out, status, run_timestamp
    FROM workspace.taxi.pipeline_log
    ORDER BY run_timestamp ASC
""").show(truncate=False)

=== Gold #1: Revenue by Pickup Borough ===
+-------------+-----------+-------------+--------+-----------------------+
|borough      |total_trips|total_revenue|avg_fare|avg_trip_distance_miles|
+-------------+-----------+-------------+--------+-----------------------+
|Manhattan    |2462272    |5.605488285E7|14.85   |2.26                   |
|Queens       |258883     |1.858396723E7|52.5    |12.55                  |
|Brooklyn     |18116      |611632.32    |29.44   |5.79                   |
|Unknown      |9586       |267453.64    |19.59   |3.16                   |
|Bronx        |5696       |213522.26    |34.16   |7.02                   |
|N/A          |1254       |145843.46    |102.66  |2.91                   |
|EWR          |276        |30944.16     |95.07   |1.02                   |
|Staten Island|44         |2847.15      |51.25   |5.16                   |
+-------------+-----------+-------------+--------+-----------------------+

=== Gold #2: Top 20 Routes by Trip Count ===
+----------